# V-JEPA × HLLSet Cortex — Pipeline Validation

Mirrors `01_ocr_hllset_pipeline_real.ipynb` from `hllset_cortex`, with the
DeepSeek-OCR encoder replaced by the **V-JEPA ViT-L/16 encoder** and the OCR
decoder replaced by the **V-JEPA predictor**.

The EWM narrative is `docs/EWM/main.tex`; the principle-level mapping is
`docs/NARRATIVE.md`.  This notebook walks the data boundary:

```text
video -> V-JEPA encoder -> z (continuous)
      -> quantize -> "tid42 tid671 ..."
      -> HLLSet lattice (gate ∩, LUT, materialize)
      -> dequantize -> z_hat
      -> V-JEPA predictor -> prediction
```

In [1]:
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "ewm_jepa").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from ewm_jepa import JEPAPipeline, Quantizer
import hllset_py
from hllset_cortex import HLLSetFilter
from hllset_cortex.domain import encoding_tokenizer, hllset_from_ids, tid

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


cuda: True NVIDIA GeForce RTX 3060


## 1. Tokenizer — 3-gram structural encoding

The same `hllset-dsl` tokenizer that processed `tid{n}` streams in
ds-hllset-cortex processes quantized V-JEPA patch IDs here.  n-grams are the
**bootstrap family** (Principle 3, Ashby–Bootstrap): each order is a distinct
measurement view of the same patch sequence.

In [2]:
tok = encoding_tokenizer()
toks = tok.tokenize(b"tid42 tid671 tid42 tid99")
print([t.decode() for t in toks])


['tid42', 'tid671', 'tid42', 'tid99', 'tid42\x00tid671', 'tid671\x00tid42', 'tid42\x00tid99', 'tid42\x00tid671\x00tid42', 'tid671\x00tid42\x00tid99']


## 2. HLLSet — IICA properties

Idempotent (same IDs, same HLLSet), Immutable (never changes), Content-
Addressed (the SHA-1 key *is* the address).  These are Principles 1 and 2 of
`main.tex`.

In [3]:
h1 = hllset_py.HLLSet.from_tokens(["tid42", "tid671", "tid42"])
h2 = hllset_py.HLLSet.from_tokens(["tid42", "tid671", "tid42"])
print("idempotent:", h1.content_key() == h2.content_key())
print("popcount  :", h1.popcount(), "->", h1.popcount(), "(immutable)")
print("CID       :", h1.content_key())


idempotent: True
popcount  : 2 -> 2 (immutable)
CID       : h:fc47e12250686bb56254d87f6f2d85b59f994698


## 3. gate_TF HLLSet — the codebook vocabulary as a content-addressed gate

In ds-hllset-cortex the gate was the decoder's BPE vocabulary.  Here it is the
**quantizer codebook**: the set of `tid{n}` the system can express in V-JEPA
latent space.  Intersection with the gate filters invalid bit positions at the
lattice level (the "world view", `main.tex` appendix).

In [4]:
CODEBOOK_SIZE = 2048
gate = hllset_from_ids(range(CODEBOOK_SIZE))
print("gate popcount:", gate.popcount())
print("gate CID     :", gate.content_key())


gate popcount: 1549
gate CID     : h:3922bd5717163e103b9c86b84c81b43cd87da30f


## 4. HLLSetFilter — LUT + TF-ranked materialization

The LUT accumulates TF for **every** observed tid — pre-gate, monotonic, and
never reset.  Materialization returns the highest-TF token at each active bit
position.  This is the Type-1 forward model of the perceptron taxonomy.

In [5]:
filt = HLLSetFilter()
filt.tokenizer = encoding_tokenizer()
res = filt.process_text("tid42 tid671 tid42 tid99")
print("restored :", res.token_strings)
print("stats    :", res.stats)


restored : ['tid671\x00tid42\x00tid99', 'tid99', 'tid42\x00tid99', 'tid42', 'tid671\x00tid42', 'tid42\x00tid671\x00tid42', 'tid671', 'tid42\x00tid671']
stats    : FilterStats(input_tokens=4, hllset_popcount=8, gate_popcount=8, output_tokens=8, compression_ratio=2.0, roundtrip_match=8, roundtrip_total=8)


## 5. Multi-document learning

The LUT is shared across streams: TF earned in one observation improves
disambiguation in later ones.  This is the cold-start rule of `main.tex`
Appendix D — TF is earned through experience, never seeded.

In [6]:
for stream in ["tid42 tid671 tid42", "tid42 tid99 tid7 tid671"]:
    filt.process_text(stream)
print(filt.summary())


{'streams': 3, 'lut_size': 13, 'lut_positions': 13, 'gate_popcount': 0, 'avg_input_tokens': 3.6666666666666665, 'avg_hllset_popcount': 7.333333333333333, 'avg_gate_popcount': 7.333333333333333, 'avg_output_tokens': 7.333333333333333, 'avg_roundtrip_match': 1.0}


## 6. Cross-document similarity — BSS

The Bell State Similarity is a directed, two-component measure
(`main.tex` appendix): τ = inclusion, ρ = exclusion.

In [12]:
ha = hllset_py.HLLSet.from_tokens(["tid42", "tid671", "tid99", "tid12345"])
hb = hllset_py.HLLSet.from_tokens(["tid42", "tid99", "tid671"])

def bss_tau(a, b):
    return a.intersection(b).popcount() / max(b.popcount(), 1)

def bss_rho(a, b):
    return a.difference(b).popcount() / max(b.popcount(), 1)

print("tau(A->B):", round(bss_tau(ha, hb), 3),
      "rho(A->B):", round(bss_rho(ha, hb), 3))
print("tau(B->A):", round(bss_tau(hb, ha), 3),
      "rho(B->A):", round(bss_rho(hb, ha), 3))


tau(A->B): 1.0 rho(A->B): 0.333
tau(B->A): 0.75 rho(B->A): 0.0


## 7. JEPAPipeline — the full boundary

This is the ewm-jepa loop: encode → quantize → HLLSet lattice → dequantize →
predict.  We compare the vanilla prediction against the prediction made from
lattice-restored context encodings.

In [13]:
pipe = JEPAPipeline(codebook_size=CODEBOOK_SIZE, seed=0)
video, masks_enc, masks_pred = pipe.make_masks(batch_size=1)
print("video:", tuple(video.shape), "mask pairs:", len(masks_enc))

result = pipe.roundtrip(video, masks_enc, masks_pred)
for i, s in enumerate(result.stats):
    print(f"pair {i}: ids {s.input_ids} (unique {s.unique_input_ids}) "
          f"-> gated set {s.gated_ids} (set retention {s.set_retention:.3f}) "
          f"| ordered {s.restored_ids} (retention {s.retention:.3f})")
    print(f"        HLLSet {s.hllset_popcount} bits -> gate {s.gate_popcount} bits "
          f"| LUT {s.lut_size} | quant cos {s.quant_cosine:.3f} "
          f"| pred cos {s.pred_cosine:.3f}")


INFO:root:MultiMaskWrapper(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed3D(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (blocks): ModuleList(
      (0-23): 24 x Block(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=Tr

/home/alexmy/.julia/conda/3/x86_64/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


pair 0: ids 616 (unique 143) -> gated set 124 (set retention 0.867) | ordered 11 (retention 0.018)
        HLLSet 1025 bits -> gate 506 bits | LUT 1228 | quant cos 0.017 | pred cos 0.393
pair 1: ids 240 (unique 94) -> gated set 94 (set retention 1.000) | ordered 35 (retention 0.146)
        HLLSet 505 bits -> gate 269 bits | LUT 1668 | quant cos 0.018 | pred cos 0.510


## 8. Latent vocabulary — TF survives gate changes

An out-of-codebook tid is filtered by the gate but still accumulates TF in the
LUT.  When the gate expands (codebook v2), the previously latent tid
materializes **immediately at its earned TF** — belief revision without cold
start (`main.tex` appendix, "Change the Vocabulary, Change the World").

In [14]:
def unigrams(res):
    return [t for t in res.token_strings if "\x00" not in t]

f2 = HLLSetFilter()
f2.tokenizer = encoding_tokenizer()
gate_v1 = hllset_from_ids(range(1000))        # narrow gate
f2.gate_hllset = gate_v1

r1 = f2.process_text("tid42 tid1500 tid42")   # tid1500 is outside gate_v1
print("narrow gate restored  :", unigrams(r1))
print("LUT knows tid1500     :",
      any(t == "tid1500" for t, _ in f2.lut.ranked_tokens()))

gate_v2 = hllset_from_ids(range(2000))        # expanded gate
f2.gate_hllset = gate_v2
r2 = f2.process_text("tid42 tid1500 tid42")
print("expanded gate restored:", unigrams(r2))


narrow gate restored  : ['tid42']
LUT knows tid1500     : True
expanded gate restored: ['tid1500', 'tid42']


## 9. Full roundtrip — summary

**What the numbers show.** The gated set pass retains the encoding vocabulary
with high fidelity (set retention → 1.0): the EWM lattice is a faithful
*set memory* over V-JEPA encodings.  The De Bruijn ordered pass degrades when
the same tid recurs many times in a long context (ordered retention is low):
**order restoration under repetition is the open problem** — exactly the
Ashby–Bootstrap trade-off that deeper n-gram bootstraps are designed to
resolve.  Even so, the predictor's output from restored context stays
correlated with the vanilla prediction (pred cos > 0.3), demonstrating that
the JEPA forward model tolerates the lattice boundary.